# Lab 27-01 · Closing the Budgets

See the [lab handout](README.md) for the background and the expected results. Each part asks you to *predict* before you run.

In [1]:
import math
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import sysbudget as s

## Part 1 · A better boost converter

*Predict:* the minutes gained when the display's boost converter improves from 85% to 92% in Example 27-01.1.

In [2]:
ldo = s.ldo_efficiency(1.8, 3.7)
base = [("SoC", 1.8, 0.88), ("DRAM", 0.4, 0.90), ("display", 1.2, 0.85), ("radio", 0.6, 0.90), ("sensors", 0.2, ldo)]
better = [row if row[0] != "display" else ("display", 1.2, 0.92) for row in base]
usable = 0.9 * 5.0 * 3.85
t0, t1 = s.power_tree(base), s.power_tree(better)
h0, h1 = s.runtime_h(usable, t0["p_batt"]), s.runtime_h(usable, t1["p_batt"])
print(f"battery power {t0['p_batt']:.3f} -> {t1['p_batt']:.3f} W (saves {t0['p_batt'] - t1['p_batt']:.3f} W)")
print(f"run time {h0:.3f} -> {h1:.3f} h, {60 * (h1 - h0):.1f} minutes more")
for name, p, eta, pin, loss in t0["rows"]:
    print(f"{name:8s} load {p:.2f} W, battery {pin:.3f} W, loss {loss:.3f} W")

battery power 4.979 -> 4.872 W (saves 0.107 W)
run time 3.479 -> 3.556 h, 4.6 minutes more
SoC      load 1.80 W, battery 2.045 W, loss 0.245 W
DRAM     load 0.40 W, battery 0.444 W, loss 0.044 W
display  load 1.20 W, battery 1.412 W, loss 0.212 W
radio    load 0.60 W, battery 0.667 W, loss 0.067 W
sensors  load 0.20 W, battery 0.411 W, loss 0.211 W


## Part 2 · Burst duration against power

*Predict:* how long a 12 W burst lasts from a 2 W steady state (Example 27-01.2: $\theta$ = 4.0 °C/W, $C_{th}$ = 60 J/°C, 25 °C air, 43 °C limit).

In [3]:
theta, cth, ta, tmax = 4.0, 60.0, 25.0, 43.0
t0 = ta + 2.0 * theta
print(f"12 W burst lasts {s.time_to_limit(t0, tmax, ta, 12.0, theta, cth):.1f} s; sustained power {s.sustained_power(tmax, ta, theta):.2f} W")
powers = np.linspace(5, 15, 101)
dur = [s.time_to_limit(t0, tmax, ta, p, theta, cth) for p in powers]
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(powers, dur)
ax.set_xlabel("burst power (W)"); ax.set_ylabel("burst duration (s)"); ax.set_ylim(0, 600)
plt.show()

12 W burst lasts 69.0 s; sustained power 4.50 W


## Part 3 · Batching to the ridge

*Predict:* the batch size at which decode on Example 27-01.3's accelerator becomes compute bound.

In [4]:
peak, bw = 400e12, 3.2e12
print(f"ridge {s.ridge(peak, bw):.0f} FLOP/byte -> batch {s.ridge(peak, bw):.0f} at 1 FLOP/byte per user")
batches = np.arange(1, 257)
att = [s.roofline(peak, bw, b * 1.0) / 1e12 for b in batches]
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(batches, att)
ax.set_xlabel("batch size (users)"); ax.set_ylabel("attainable TFLOP/s")
plt.show()

ridge 125 FLOP/byte -> batch 125 at 1 FLOP/byte per user


## Part 4 · What a watt is worth

*Predict:* the exact and linear performance gains if power gating saves a full watt in Example 27-01.6 (dynamic budget 3.0 W).

In [5]:
saved = np.linspace(0, 1.5, 61)
exact = [100 * (s.perf_under_cap(3.0 + x, 3.0) - 1) for x in saved]
linear = [100 * x * s.shadow_price_per_watt(3.0) for x in saved]
print(f"1 W saved: exact {100 * (s.perf_under_cap(4.0, 3.0) - 1):.2f} %, linear {100 * s.shadow_price_per_watt(3.0):.1f} %")
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(saved, exact, label="exact (cube root)")
ax.plot(saved, linear, "--", label="linear (shadow price)")
ax.set_xlabel("static power saved (W)"); ax.set_ylabel("performance gain (%)"); ax.legend()
plt.show()

1 W saved: exact 10.06 %, linear 11.1 %
